# Is There a Pattern? Global Moran's I

**Series:** Spatial Autocorrelation with PySAL (notebook 2 of 3)
**Time:** about 25 minutes

**In this notebook you will:**
- Map a variable and ask whether similar values sit together
- Compute Global Moran's I for median age
- Read the p-value from a permutation test
- Interpret a Moran scatterplot

**You should already know:** notebook 02a (spatial weights).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## The question

Do counties with an older population tend to sit next to other older counties?
**Global Moran's I** answers this with one number for the whole map:

| Moran's I | Meaning |
|---|---|
| Clearly above 0 | Similar values cluster together (positive autocorrelation) |
| Close to 0 | No spatial pattern; values look randomly placed |
| Below 0 | Neighbors tend to be different (a checkerboard) |

## Step 1: Install and import

In [ ]:
!pip install -q libpysal esda splot mapclassify

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests

## Step 2: Get the data

We use the **American Community Survey (ACS) 5-year estimates, 2023** for Indiana's 92 counties:

| Census variable | Meaning | Our column |
|---|---|---|
| `B01003_001E` | Total population | `total_pop` |
| `B01002_001E` | Median age | `median_age` |
| `B02001_005E` | Asian alone population | `asian_pop` |

> **Every learner needs their own free Census API key.** It is the same key used in the Census data
> session. If you do not have one yet, request it at <https://api.census.gov/data/key_signup.html>;
> it arrives by email within a few minutes and must be activated from the link in that email.

Paste your key between the quotes below. The data and the county boundaries are read straight from
the Census website into memory, so there is nothing to download.

In [ ]:
CENSUS_KEY = "PASTE_YOUR_CENSUS_API_KEY_HERE"

def load_indiana_counties(key):
    if key.startswith("PASTE"):
        raise ValueError("Paste your free Census API key into CENSUS_KEY above, then run this cell again.")
    # 1. Attributes from the Census API
    reply = requests.get(
        "https://api.census.gov/data/2023/acs/acs5",
        params={
            "get": "NAME,B01003_001E,B01002_001E,B02001_005E",
            "for": "county:*",
            "in": "state:18",          # 18 = Indiana
            "key": key,
        },
        timeout=60,
    )
    reply.raise_for_status()
    rows = reply.json()
    table = pd.DataFrame(rows[1:], columns=rows[0])
    table["GEOID"] = table["state"] + table["county"]
    table = table.rename(columns={"B01003_001E": "total_pop",
                                  "B01002_001E": "median_age",
                                  "B02001_005E": "asian_pop"})
    for col in ["total_pop", "median_age", "asian_pop"]:
        table[col] = pd.to_numeric(table[col])
    table["pct_asian"] = 100 * table["asian_pop"] / table["total_pop"]

    # 2. County boundaries from the Census cartographic boundary files
    shapes = gpd.read_file(
        "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip")
    shapes = shapes[shapes["STATEFP"] == "18"][["GEOID", "geometry"]]

    # 3. Join, then project to meters (UTM zone 16N) for spatial analysis
    counties = shapes.merge(table, on="GEOID").to_crs(epsg=26916)
    return counties.reset_index(drop=True)

counties = load_indiana_counties(CENSUS_KEY)
print(len(counties), "counties loaded")
counties[["NAME", "total_pop", "median_age", "pct_asian"]].head()

## Step 3: Map median age

In [ ]:
counties.plot(column="median_age", scheme="Quantiles", k=5, cmap="YlOrRd",
              legend=True, edgecolor="white", figsize=(6, 8))
plt.title("Median age by county (5 quantile classes)")
plt.axis("off")
plt.show()

Looking at a map, our eyes find patterns everywhere. Moran's I tells us whether the pattern is
stronger than chance.

## Step 4: Build Queen weights

In [ ]:
from libpysal.weights import Queen

w = Queen.from_dataframe(counties, use_index=True)
w.transform = "R"

## Step 5: Compute Moran's I

`permutations=999` shuffles the values across counties 999 times to see what Moran's I looks like
when there is no pattern. The **p-value** (`p_sim`) is the share of shuffles that were as extreme
as the real map.

In [ ]:
from esda.moran import Moran

mi = Moran(counties["median_age"], w, permutations=999)
print(f"Moran's I: {mi.I:.3f}")
print(f"Expected I with no pattern: {mi.EI:.3f}")
print(f"p-value: {mi.p_sim:.3f}")

**How to read it:** if the p-value is below 0.05 and I is positive, older counties really do
cluster near other older counties.

## Step 6: The Moran scatterplot

Each dot is a county. The x-axis is its own (standardized) median age; the y-axis is the average of
its neighbors (the **spatial lag**). The slope of the red line is Moran's I.

- Top right: old county, old neighbors (**High-High**)
- Bottom left: young county, young neighbors (**Low-Low**)
- The other two corners are counties unlike their neighbors

In [ ]:
from splot.esda import moran_scatterplot

fig, ax = moran_scatterplot(mi, aspect_equal=True)
ax.set_xlabel("Median age (standardized)")
ax.set_ylabel("Neighbors' median age (spatial lag)")
plt.show()

## Step 7: The permutation test, visually

The gray curve shows Moran's I from 999 shuffled maps. The red line is the real value.
If the red line sits far outside the curve, the pattern is not chance.

In [ ]:
from splot.esda import plot_moran_simulation

plot_moran_simulation(mi)
plt.show()

## Try it yourself

Compute Moran's I for **percent Asian** (`pct_asian`). Is it more or less clustered than median age? What might explain that?

*Hint:* think about where Indiana's largest cities and universities are.

<details>
<summary><b>Show a solution</b></summary>

```python
mi_asian = Moran(counties["pct_asian"], w, permutations=999)
print(f"I = {mi_asian.I:.3f}, p = {mi_asian.p_sim:.3f}")
```

</details>

In [ ]:
# Write your answer here

## Recap

- Global Moran's I gives one number for how clustered a variable is.
- Positive I with p below 0.05 means similar values sit together.
- The Moran scatterplot compares each place to its neighbors' average.
- Global Moran's I says *whether* there is clustering, not *where*.

**Next:** [Local Moran's I and Hotspots](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/02c_Local_Morans_I_Hotspots.ipynb)